In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
from sklearn.datasets import make_moons

In [ ]:
X, y = make_moons(n_samples=1000, noise=0.1)
X = torch.tensor(X, dtype=torch.float32).view(-1, 2)
y = torch.tensor(y, dtype=torch.float32)

### $\beta$ scheduling

In diffusion models, the $\beta$ scheduling refers to the way the noise level is controlled throughout the diffusion process. The $\beta$ values are typically defined for each time step in the diffusion process, and they determine how much noise is added at each *forward diffusion step* according to the following equation:

$$
\begin{equation}
\begin{aligned}
    \mathbf{q} \left( \mathbf{x}_{t} | \mathbf{x}_{t-1} \right)
    &=
    \mathcal{N} \left( \mathbf{x}_{t} : \sqrt{1 - \beta_{t}} \mathbf{x}_{t-1}, \beta{t} \mathbf{I} \right)\,, \\
\end{aligned}
\end{equation}
$$

where $\mathbf{x}_{t}$ is the noisy image at time step $t$, $\mathbf{x}_{t-1}$ is the image at the previous time step, and $\beta_{t}$ is the noise level for that time step. Essentially, **$\beta_{t}$ is the variance of the Gaussian noise added at each step**.

The original DDPM paper defined a so-called *linear $\beta$ schedule* as follows:

$$
\begin{equation}
    \beta_{t}
    =
    \beta_{0} + \frac{t - 1}{T - 1} \cdot \left( \beta_{T} - \beta_{0} \right)\,,
\end{equation}
$$

where $T$ is the total number of diffusion steps, $\beta_{0}$ is the initial noise level, and $\beta_{T}$ is the final noise level. Ho et al. (2020) choose the values $\beta_{0} = 10^{-4}$ and $\beta_{T} = 2 \cdot 10^{-2}$ with $T = 1000$, which results in a linear increase of the noise level between these two values over the entire diffusion process, i.e. a $1000$ steps.

In [ ]:
def beta_linear(t, T=1000, beta_0=1e-4, beta_T=2e-2):
    '''Linear schedule for beta values from Ho et al. (2020).'''
    return beta_0 + (t-1)/(T-1) * (beta_T - beta_0)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 4), dpi=120)

beta_0, beta_T = 1e-4, 2e-2
T = 1000
t = torch.arange(1, T+1, dtype=torch.float32)
beta_t = beta_linear(t, T, beta_0, beta_T)
ax.plot(t.numpy(), beta_t.numpy(), color='0.3', lw=3)
ax.set_title(r'Linear $\beta(t)$ schedule', loc='left', fontsize=10)
ax.axhline(beta_0, color='tab:red', ls='--', lw=1)
ax.text(T/4, beta_0, rf'$\beta_0$ = {beta_0:.3e}', color='tab:red', fontsize=10,
        va='bottom', ha='left')
ax.axhline(beta_T, color='tab:green', ls='--', lw=1)
ax.text(T/4, beta_T, rf'$\beta_T$ = {beta_T:.3e}', color='tab:green', fontsize=10,
        va='top', ha='left')

plt.show()

### Reparameterization trick

To practically implement the forward step defined by the conditional distribution $q(\mathbf{x}_t | \mathbf{x}_{t-1})$, we can use the so-called *reparameterization trick*. This allows us to express $\mathbf{x}_t$ in a computable form. For the convenience of our notation, a new variable $\alpha_t$ is introduced, defined simply as $\alpha_t = 1 - \beta_t$.

Using this definition, the equation to sample $\mathbf{x}_t$ from $\mathbf{x}_{t-1}$ is written as:

$$
\begin{equation}
    \mathbf{x}_{t}
    =
    \sqrt{\alpha_t} \mathbf{x}_{t-1}
    +
    \sqrt{1 - \alpha_t} \boldsymbol{\varepsilon}
\end{equation}    
$$

where $\boldsymbol{\varepsilon}$ is a random noise term sampled from a standard Gaussian distribution, $\boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$. In this form, it is clear that $\sqrt{\alpha_t}$ scales the signal from the previous step ($\mathbf{x}_{t-1}$), while the term $\sqrt{1 - \alpha_t}$ (which is equivalent to $\sqrt{\beta_t}$) scales the newly introduced noise. Speaking again in more demonstrative terms, $\alpha_t$ indicates how much of the original signal is retained at each step.

In [ ]:
def beta_linear(t, T=1000, beta_0=1e-4, beta_T=2e-2):
    '''
    Linear schedule for beta values from Ho et al. (2020) using the
    reparametrization trick.
    '''
    beta_t = beta_0 + (t-1)/(T-1) * (beta_T - beta_0)
    alpha_t = 1 - beta_t
    return beta_t, alpha_t

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 4), dpi=120)

beta_0, beta_T = 1e-4, 2e-2
T = 1000
t = torch.arange(1, T+1, dtype=torch.float32)
beta_t, alpha_t = beta_linear(t, T, beta_0, beta_T)
ax.plot(t.numpy(), beta_t.numpy(), color='0.3', lw=3)
ax.text(T/4, beta_t[-1], rf'$\beta(t)$', color='0.3', fontsize=10,
        va='bottom', ha='left')
ax.plot(t.numpy(), alpha_t.numpy(), color='tab:red', lw=3)
ax.text(T/4, alpha_t[-1], rf'$\alpha(t)$', color='tab:red', fontsize=10,
        va='top', ha='left')
ax.set_title(r'Linear $\beta(t)$ schedule reparametrization', loc='left', fontsize=10)

plt.show()

### Cosine scheduling

While the linear $\beta_t$ schedule introduced in the original DDPM paper works reasonably well, subsequent works have observed that linear schedules may not optimally control the signal-to-noise ratio throughout the diffusion process. In particular:
- At early timesteps, the noise is injected very slowly, leading to unnecessary redundancy.
- At later timesteps, too much noise is injected too quickly, making it harder for the model to recover the data during the reverse diffusion process (i.e. denoising).

To address these shortcomings, several alternative schedule have been proposed. One particularly effective approach is the *cosine schedule*, proposed by Nichol & Dhariwal (2021).

Instead of directly defining $\beta_t$, the cosine schedule defines a sequence of cumulative products of $\alpha_t$, usually denoted $\bar{\alpha}t$, which directly translates to how much of the original signal survives after $t$ steps. The idea is to smoothly decay $\bar{\alpha}_t$ as a function of $t$ using a cosine curve. The schedule is defined as follows:

$$
\begin{equation}
    \bar{\alpha}_t
    =
    \cos^{2} \left( \frac{t/T + s}{1 + s} \cdot \frac{\pi}{2} \right)\,,
\end{equation}
$$

where $s$ is a scaling factor that controls the steepness of the curve, typically set to $0.008$, and with variance $\bar{\beta}_{t}$ defined as:

$$
\begin{equation}
    \bar{\beta}_{t}
    =
    1 - \frac{\bar{\alpha}_{t}}{\bar{\alpha}_{t} - 1}\,.
\end{equation}
$$

In [ ]:
def beta_cosine(t, T=1000, s=0.008):
    '''Cosine schedule for beta values from Nichol & Dhariwal (2021).'''
    alpha_bar = torch.cos((t/T + s)/(1 + s) * np.pi/2)**2
    beta_bar = 1 - (alpha_bar[1:] / alpha_bar[:-1])
    return alpha_bar, beta_bar

In [ ]:
def beta_linear(t):
    '''
    Parameters
    ----------
    t : int
        Number of time steps.

    Returns
    -------
    beta_tilde : torch.Tensor
        The cumulative noise for the linear beta schedule.
    '''
    beta_t = torch.linspace(1e-4, 0.02, t)
    alpha_t = torch.cumprod(1 - beta_t, dim=0)
    beta_tilde = (1 - alpha_t[:-1]) / (1 - alpha_t[1:]) * beta_t[1:]
    # alpha_t = alpha.gather(0, t).view(-1, *[1]*(X.dim() - 1))
    return beta_tilde

In [ ]:
def beta_cosine(t, s=0.008):
    ti = torch.arange(0, t+1, dtype=torch.int32)
    ft = torch.cos(((ti / t) + s) / (1 + s) * np.pi / 2)**2
    alpha_t = ft[:-1]/ft[0]
    beta_t = 1 - (alpha_t[1:] / alpha_t[:-1])
    return torch.clamp(beta_t, 0.001, 0.999)

In [ ]:
t_fw = 30

nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*4, nr*4), dpi=120)
fig.subplots_adjust(wspace=0.3, hspace=0.2)

ax = axes[0]
beta = beta_linear(t_fw)
ax.plot(beta.numpy(), 'o-')

ax = axes[1]
beta = beta_cosine(t_fw, s=0.008)
ax.plot(beta.numpy(), 'o-')

plt.show()

In [ ]:
def forward_diffusion(x0, beta_t):
    '''
    Forward diffusion process with the reparametrization trick.

    Parameters:
    -----------
    x0 : torch.Tensor
        The initial data point (shape: [batch_size, num_features]).
    beta_t : float
        The noise level at time t (0 <= beta_t <= 1).

    Returns:
    --------
    xt : torch.Tensor
        The noisy data point at time t (shape: [batch_size, num_features]).
    noise : torch.Tensor
        The noise added to the data point (shape: [batch_size, num_features]).
    '''
    # Generate random Gaussian noise
    noise = torch.randn_like(x0)
    # Using the reparametrization trick apply the Markow diffusion kernel
    # to the initial data point x0
    xt = torch.sqrt(1 - beta_t) * x0 + torch.sqrt(beta_t) * noise
    return xt, noise

In [ ]:
t_fw = np.arange(0, 101, 20, dtype=int)[1:]

nr, nc = 2, t_fw.size+1
fig, axes = plt.subplots(nr, nc, figsize=(nc*4, nr*4), dpi=120)
fig.subplots_adjust(wspace=0.2, hspace=0.2)

for ax in axes.flat:
    ax.set_aspect(1)
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)

# Initial state
ax = axes[0][0]
ax.scatter(*X.numpy().T, c=y.numpy(), s=10, cmap='viridis')
ax.set_title(f"Number of passes: 0")
axes[1][0].set_visible(False)

# Forward diffusion process
for ax, (i, ti) in zip(axes.T[1:], enumerate(t_fw)):
    beta = torch.linspace(1e-4, 0.02, ti)  # Linear schedule from the OG paper
    alpha = torch.cumprod(1 - beta, dim=0)

    t = torch.randint(0, ti, (X.size(0),), device=X.device, dtype=torch.long)
    alpha_t = alpha.gather(0, t).view(-1, *[1]*(X.dim() - 1))
    beta_t = 1 - alpha_t
    xt, noise = forward_diffusion(X, beta_t)

    ax[0].scatter(*xt.numpy().T, c=y.numpy(), s=10, cmap='viridis')
    ax[0].set_title(f"Number of passes: {ti}")

    ax[1].scatter(*noise.numpy().T, c=y.numpy(), s=10, cmap='viridis')
    ax[1].set_title(f"Noise at t={ti}")

plt.show()